In [1]:
# --------------------------------------------------
# 1. Imports (add these to the existing ones)
# --------------------------------------------------
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.preprocessing import MultiLabelBinarizer
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import itertools
from torchvision.transforms import Resize
from ast import literal_eval
from sklearn.metrics import matthews_corrcoef
from torchvision import models
from transformers import AutoTokenizer, AutoModelForMaskedLM
import math
import json
from collections import defaultdict
import seaborn as sns
from matplotlib.colors import LogNorm
import random
# --------------------------------------------------

Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu128 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info


this is for all_endpoint + non_functional

In [2]:
with open('cleaned_clustered_dataset.json') as f:
    clustered_dataset = json.load(f)



In [3]:
class_count = defaultdict(int)
for cluster in clustered_dataset:
    for s in cluster:
        for c in s['classes']:
            class_count[c] += 1

class_count

defaultdict(int,
            {'other-functional': 7040,
             'metabolic': 11589,
             'inhibitor': 2046,
             'toxic': 22440,
             'anti-cancer': 12390,
             'drug-delivery': 2202,
             'anti-bacterial': 29969,
             'anti-fungal': 12747,
             'anti-parasitic': 6586,
             'anti-viral': 6894,
             'signal-peptide': 22226,
             'immunological': 4413,
             'cell-cell-communication': 3249,
             'non-functional': 163115})

In [4]:
# cleaning clusters , removing clustures with empty sequences or empty classes
removes_seqs = 0
for i, cluster in enumerate(clustered_dataset):
    new_cluster = []
    for entry in cluster:
        if entry['sequence'] == '' or len(entry['classes']) == 0:
            removes_seqs += 1
        else:
            new_cluster.append(entry)

    clustered_dataset[i] = new_cluster

unfiltured_clusters_size = len(clustered_dataset)
clustered_dataset = [cluster for cluster in clustered_dataset if len(cluster) > 0]
print(f"Removed {unfiltured_clusters_size - len(clustered_dataset)} clusters")
print(f"Removed {removes_seqs} sequences")

Removed 0 clusters
Removed 0 sequences


# dividing into train and test


In [5]:
train_val_test_ratio = [0.7, 0.1, 0.2]
random_state = 42

if len(train_val_test_ratio) != 3:
    raise ValueError("train_val_test_ratio must contain [train, val, test].")

ratio_sum = sum(train_val_test_ratio)
if ratio_sum <= 0:
    raise ValueError("Sum of train_val_test_ratio must be > 0.")

train_ratio, val_ratio, test_ratio = [r / ratio_sum for r in train_val_test_ratio]

all_seqs_count = sum(len(cluster) for cluster in clustered_dataset)
print('all_seqs_count', all_seqs_count)

train_size = int(all_seqs_count * train_ratio)
val_size = int(all_seqs_count * val_ratio)
test_size = all_seqs_count - train_size - val_size

train_seqs = []
val_seqs = []
test_seqs = []

train_cluster_count = 0
val_cluster_count = 0
test_cluster_count = 0

all_cluster_ids = [i for i in range(len(clustered_dataset))]
rng = random.Random(random_state)
rng.shuffle(all_cluster_ids)

for cluster_id in all_cluster_ids:
    cluster = clustered_dataset[cluster_id]
    if len(train_seqs) < train_size:
        train_seqs.extend(cluster)
        train_cluster_count += 1
    elif len(val_seqs) < val_size:
        val_seqs.extend(cluster)
        val_cluster_count += 1
    else:
        test_seqs.extend(cluster)
        test_cluster_count += 1

print('train_seqs', len(train_seqs))
print('val_seqs', len(val_seqs))
print('test_seqs', len(test_seqs))

print('train_cluster_count', train_cluster_count)
print('val_cluster_count', val_cluster_count)
print('test_cluster_count', test_cluster_count)

all_seqs_count 257812
train_seqs 180471
val_seqs 25781
test_seqs 51560
train_cluster_count 87893
val_cluster_count 12557
test_cluster_count 25276


# creating dataset

In [6]:
# other-functional is present with known functionl class then remove other-functional
for item in train_seqs:
    if 'other-functional' in item['classes'] and len(item['classes']) > 1:
        item['classes'].remove('other-functional')
        if len(item['classes']) == 0: raise Exception("no class left")

for item in test_seqs:
    if 'other-functional' in item['classes'] and len(item['classes']) > 1:
        item['classes'].remove('other-functional')
        if len(item['classes']) == 0: raise Exception("no class left")

for item in val_seqs:
    if 'other-functional' in item['classes'] and len(item['classes']) > 1:
        item['classes'].remove('other-functional')
        if len(item['classes']) == 0: raise Exception("no class left")



In [7]:
all_endpoints = set([c for e in train_seqs+test_seqs+val_seqs for c in e['classes']]) 

sorted(list(all_endpoints))

['anti-bacterial',
 'anti-cancer',
 'anti-fungal',
 'anti-parasitic',
 'anti-viral',
 'cell-cell-communication',
 'drug-delivery',
 'immunological',
 'inhibitor',
 'metabolic',
 'non-functional',
 'other-functional',
 'signal-peptide',
 'toxic']

In [8]:
# endpoints = [
#  'anti-cancer',
#  'anti-fungal',
#  'anti-parasitic',
#  'anti-viral',
#  'non-functional'
#  ]

endpoints = ['anti-bacterial',
 'anti-cancer',
 'anti-fungal',
 'anti-parasitic',
 'anti-viral',
 'cell-cell-communication',
 'drug-delivery',
 'immunological',
 'inhibitor',
 'metabolic',
 'non-functional',
 'other-functional',
 'signal-peptide',
 'toxic']

 
endpoints_set = set(endpoints)
endpoint_index = {endpoint: i for i, endpoint in enumerate(endpoints)}
index_endpoint = {i: endpoint for i, endpoint in enumerate(endpoints)}


In [9]:
endpoints_set

{'anti-bacterial',
 'anti-cancer',
 'anti-fungal',
 'anti-parasitic',
 'anti-viral',
 'cell-cell-communication',
 'drug-delivery',
 'immunological',
 'inhibitor',
 'metabolic',
 'non-functional',
 'other-functional',
 'signal-peptide',
 'toxic'}

In [10]:
train_df_rows = []
for entry in train_seqs:
    row = [False]*len(endpoints)
    if len(set(entry['classes']).intersection(endpoints_set)) > 0: 
        
        for c in entry['classes']:
            if c in endpoints:
                row[endpoint_index[c]] = True

    
        row = [entry['sequence']] + row
        train_df_rows.append(row)
    
val_df_rows = []
for entry in val_seqs:
    row = [False]*len(endpoints)
    if len(set(entry['classes']).intersection(endpoints_set)) > 0: 

        for c in entry['classes']:
            if c in endpoints:
                row[endpoint_index[c]] = True

        row = [entry['sequence']] + row
        val_df_rows.append(row)

test_df_rows = []
for entry in test_seqs:
    row = [False]*len(endpoints)
    if len(set(entry['classes']).intersection(endpoints_set)) > 0: 

        for c in entry['classes']:
            if c in endpoints:
                row[endpoint_index[c]] = True

        row = [entry['sequence']] + row
        test_df_rows.append(row)

train_df = pd.DataFrame(train_df_rows, columns=['sequence'] + endpoints)
val_df = pd.DataFrame(val_df_rows, columns=['sequence'] + endpoints)
test_df = pd.DataFrame(test_df_rows, columns=['sequence'] + endpoints)


In [11]:
summary = train_df[endpoints].apply(pd.Series.value_counts)

# 2. Calculate True Percentage and add it as a new row
# We multiply by 100 to get a readable percentage (e.g., 75.0 instead of 0.75)
summary.loc['True %'] = train_df[endpoints].mean() * 100

# Print the final summary
print(summary)

        anti-bacterial    anti-cancer   anti-fungal  anti-parasitic  \
False    159502.000000  171656.000000  171554.00000   175865.000000   
True      20969.000000    8815.000000    8917.00000     4606.000000   
True %       11.619041       4.884441       4.94096        2.552211   

           anti-viral  cell-cell-communication  drug-delivery  immunological  \
False   175475.000000            178081.000000  178847.000000  177435.000000   
True      4996.000000              2390.000000    1624.000000    3036.000000   
True %       2.768312                 1.324312       0.899868       1.682265   

            inhibitor      metabolic  non-functional  other-functional  \
False   178988.000000  172233.000000    66678.000000     178237.000000   
True      1483.000000    8238.000000   113793.000000       2234.000000   
True %       0.821739       4.564722       63.053344          1.237872   

        signal-peptide          toxic  
False    164871.000000  164737.000000  
True      15600.0

In [12]:
summary = test_df[endpoints].apply(pd.Series.value_counts)

# 2. Calculate True Percentage and add it as a new row
# We multiply by 100 to get a readable percentage (e.g., 75.0 instead of 0.75)
summary.loc['True %'] = test_df[endpoints].mean() * 100

# Print the final summary
print(summary)

        anti-bacterial   anti-cancer  anti-fungal  anti-parasitic  \
False     45293.000000  49132.000000  49041.00000    50244.000000   
True       6267.000000   2428.000000   2519.00000     1316.000000   
True %       12.154771      4.709077      4.88557        2.552366   

          anti-viral  cell-cell-communication  drug-delivery  immunological  \
False   50281.000000              51014.00000   51168.000000   50648.000000   
True     1279.000000                546.00000     392.000000     912.000000   
True %      2.480605                  1.05896       0.760279       1.768813   

           inhibitor    metabolic  non-functional  other-functional  \
False   51191.000000  49450.00000    18796.000000      50873.000000   
True      369.000000   2110.00000    32764.000000        687.000000   
True %      0.715671      4.09232       63.545384          1.332428   

        signal-peptide         toxic  
False     47144.000000  47045.000000  
True       4416.000000   4515.000000  
True

In [13]:
random.random()

0.9815284536389205

In [14]:
import torch
import numpy as np


import torch


# Example usage:
# seq = "ALDFR" -> Dipeptides: AL, LD, DF, FR
# vector = get_dipeptide_composition(seq)


esm_tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")



PCP_columns = ['PCP_PC','PCP_NC','PCP_NE','PCP_PO','PCP_NP','PCP_AL','PCP_CY','PCP_AR','PCP_AC','PCP_BS','PCP_NE_pH','PCP_HB','PCP_HL','PCP_NT','PCP_HX','PCP_SC','PCP_SS_HE','PCP_SS_ST','PCP_SS_CO','PCP_SA_BU','PCP_SA_EX','PCP_SA_IN','PCP_TN','PCP_SM','PCP_LR','PCP_Z1','PCP_Z2','PCP_Z3','PCP_Z4','PCP_Z5']
PCP_DATA = {'A': np.array([0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.24, -2.32, 0.6, -0.14, 1.3]),
'C': np.array([0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.84, -1.67, 3.71, 0.18, -2.65]),
'D': np.array([0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 3.98, 0.93, 1.93, -2.46, 0.75]),
'E': np.array([0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 3.11, 0.26, -0.11, -3.04, -0.25]),
'F': np.array([0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, -4.22, 1.94, 1.06, 0.54, -0.62]),
'G': np.array([0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 2.05, -4.06, 0.36, -0.82, -0.38]),
'H': np.array([1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 2.47, 1.95, 0.26, 3.9, 0.09]),
'I': np.array([0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, -3.89, -1.73, -1.71, -0.84, 0.26]),
'K': np.array([1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 2.29, 0.89, -2.49, 1.49, 0.31]),
'L': np.array([0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, -4.28, -1.3, -1.49, -0.72, 0.84]),
'M': np.array([0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, -2.85, -0.22, 0.47, 1.94, -0.98]),
'N': np.array([0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 3.05, 1.62, 1.04, -1.15, -1.61]),
'P': np.array([0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, -1.66, 0.27, 1.84, 0.7, 2.0]),
'Q': np.array([0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.75, 0.5, -1.44, -1.34, 0.66]),
'R': np.array([1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 3.52, 2.5, -3.5, 1.99, -0.17]),
'S': np.array([0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 2.39, -1.07, 1.15, -1.39, 0.67]),
'T': np.array([0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.75, -2.18, -1.12, -1.46, -0.4]),
'V': np.array([0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, -2.59, -2.64, -1.54, -0.85, -0.02]),
'W': np.array([0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, -4.36, 3.94, 0.59, 3.44, -1.59]),
'Y': np.array([0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, -2.54, 2.44, 0.43, 0.04, -1.47]),
'X': np.array([0.15, 0.1, 0.75, 0.25, 0.45, 0.3, 0.05, 0.15, 0.1, 0.15, 0.75, 0.5, 0.25, 0.3, 0.1, 0.1, 0.4, 0.35, 0.25, 0.4, 0.3, 0.3, 0.2, 0.45, 0.55, 0.0025000000000000356, 0.002499999999999991, 0.0019999999999999545, 0.0004999999999999877, -0.16299999999999998]),
'<pad>': np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
'<start>': np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
'<end>': np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
}

PCP_DATA_ID = dict()
for k, v in PCP_DATA.items():
    k = esm_tokenizer.token_to_id(k)
    PCP_DATA_ID[k] = v

def get_pcp_features(sequence):
    if isinstance(sequence, str):
        sequence = list(sequence)
        result = [PCP_DATA.get(c, PCP_DATA['X']) for c in sequence]
        result = torch.tensor(result, dtype=torch.float32)

    if isinstance(sequence, torch.Tensor):
        result = [PCP_DATA_ID.get(c.item(), PCP_DATA_ID[3]) for c in sequence] # <unk>
        result = torch.tensor(result, dtype=torch.float32)
    return result



In [15]:


class PeptideDatasetBalanced(Dataset):

    def __init__(
        self,
        dataframe,
        tokenizer,
        label_columns,
        max_length: int = 128
    ):
        """
        Args:
            dataframe (pd.DataFrame): DataFrame containing the data.
                                      Must have a 'sequence' column.
            tokenizer (AutoTokenizer): A Hugging Face tokenizer (e.g., for ESM-2).
            label_columns (List[str]): A list of column names that represent the labels.
            max_length (int): Maximum sequence length for padding/truncation.
        """
        self.df = dataframe
        self.tokenizer = tokenizer
        self.sequences = self.df['sequence'].values
        self.labels = self.df[label_columns].values
        self.label_columns = label_columns
        self.max_length = max_length

        functional_idxs = []
        non_functional_idxs = []
        for i, label in enumerate(self.labels):
            if label[endpoint_index['non-functional']] == True:
                non_functional_idxs.append(i)
            else:
                functional_idxs.append(i)
        
        non_functional_idxs = [non_functional_idxs[i: i+3] for i in range(0, len(non_functional_idxs), 3)]
        non_functional_idxs = [i for i in non_functional_idxs]

        functional_idxs = [functional_idxs[i: i+1] for i in range(0, len(functional_idxs), 1)]
        self.all_idxs = non_functional_idxs + functional_idxs

        # Pre-tokenize ALL sequences once
        sequences = dataframe['sequence'].tolist()
        encodings = tokenizer(
            sequences,
            add_special_tokens=True,
            max_length=max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        self.input_ids = encodings['input_ids']
        self.attention_masks = encodings['attention_mask']
        
        # Pre-compute PCP features for all sequences
        self.pcp_features = torch.stack([
            get_pcp_features(self.input_ids[i]) for i in range(len(sequences))
        ])

    def __len__(self) -> int:
        """Returns the total number of samples in the dataset."""
        return len(self.all_idxs)

    def __getitem__(self, index: int):
        """
        Retrieves a single sample from the dataset.

        Args:
            index (int): The index of the sample to retrieve.

        Returns:
            A dictionary containing:
            - 'input_ids': Token IDs of the sequence.
            - 'attention_mask': Mask to avoid performing attention on padding tokens.
            - 'labels': A multi-hot encoded tensor of labels.
            - 'img_input': A dummy tensor to match the MultiModelNetwork's forward signature.
        """
        idx = random.choice(self.all_idxs[index])

        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_masks[idx],
            'pcp': self.pcp_features[idx],
            'labels': torch.FloatTensor(self.labels[idx])
        }




class PeptideDataset(Dataset):

    def __init__(
        self,
        dataframe,
        tokenizer,
        label_columns,
        max_length: int = 128
    ):
        """
        Args:
            dataframe (pd.DataFrame): DataFrame containing the data.
                                      Must have a 'sequence' column.
            tokenizer (AutoTokenizer): A Hugging Face tokenizer (e.g., for ESM-2).
            label_columns (List[str]): A list of column names that represent the labels.
            max_length (int): Maximum sequence length for padding/truncation.
        """
        self.df = dataframe
        self.tokenizer = tokenizer
        self.sequences = self.df['sequence'].values
        self.labels = self.df[label_columns].values
        self.label_columns = label_columns
        self.max_length = max_length

        functional_idxs = []
        non_functional_idxs = []
        for i, label in enumerate(self.labels):
            if label[endpoint_index['non-functional']] == True:
                non_functional_idxs.append(i)
            else:
                functional_idxs.append(i)
        
        non_functional_idxs = [non_functional_idxs[i: i+1] for i in range(0, len(non_functional_idxs), 1)]
        non_functional_idxs = [i for i in non_functional_idxs]

        functional_idxs = [functional_idxs[i: i+1] for i in range(0, len(functional_idxs), 1)]
        self.all_idxs = non_functional_idxs + functional_idxs

        # Pre-tokenize ALL sequences once
        sequences = dataframe['sequence'].tolist()
        encodings = tokenizer(
            sequences,
            add_special_tokens=True,
            max_length=max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        self.input_ids = encodings['input_ids']
        self.attention_masks = encodings['attention_mask']
        
        # Pre-compute PCP features for all sequences
        self.pcp_features = torch.stack([
            get_pcp_features(self.input_ids[i]) for i in range(len(sequences))
        ])

    def __len__(self) -> int:
        """Returns the total number of samples in the dataset."""
        return len(self.all_idxs)

    def __getitem__(self, index: int):
        """
        Retrieves a single sample from the dataset.

        Args:
            index (int): The index of the sample to retrieve.

        Returns:
            A dictionary containing:
            - 'input_ids': Token IDs of the sequence.
            - 'attention_mask': Mask to avoid performing attention on padding tokens.
            - 'labels': A multi-hot encoded tensor of labels.
            - 'img_input': A dummy tensor to match the MultiModelNetwork's forward signature.
        """
        idx = random.choice(self.all_idxs[index])

        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_masks[idx],
            'pcp': self.pcp_features[idx],
            'labels': torch.FloatTensor(self.labels[idx])
        }

In [16]:
LABEL_COLUMNS = endpoints

MAX_LEN = 100
train_dataset = PeptideDataset(train_df, esm_tokenizer, LABEL_COLUMNS, MAX_LEN)
val_dataset = PeptideDataset(val_df, esm_tokenizer, LABEL_COLUMNS, MAX_LEN)
test_dataset = PeptideDataset(test_df, esm_tokenizer, LABEL_COLUMNS, MAX_LEN)

/tmp/ipykernel_1739686/2155206720.py:57: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  result = torch.tensor(result, dtype=torch.float32)


In [17]:
len(train_dataset),  len(val_dataset), len(test_dataset)

(180471, 25781, 51560)

In [18]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=True)

In [19]:
# Calculate weights
NON_FUNC_IDX = endpoint_index['non-functional']
FUNC_INDICES = [i for k, i in endpoint_index.items() if k != 'non-functional']

In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from transformers import AutoModelForMaskedLM

# =====================================================================
# Model v19 (Lite): Sequence-Level Cross-Attention + Task Query Decoder
# Parameter Count: ~19.6 Million
# =====================================================================

class ESM2_Encoder(nn.Module):
    def __init__(self, model_name, trainable=True, unfreeze_last_n=0):
        super().__init__()
        self.esm_mlm = AutoModelForMaskedLM.from_pretrained(model_name)
        self.hidden_size = self.esm_mlm.config.hidden_size
        if not trainable:
            for param in self.esm_mlm.parameters():
                param.requires_grad = False
            if unfreeze_last_n > 0:
                for layer in self.esm_mlm.esm.encoder.layer[-unfreeze_last_n:]:
                    for param in layer.parameters():
                        param.requires_grad = True

    def forward(self, input_ids, attention_mask):
        return self.esm_mlm.esm(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state

class SEBlock(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction), nn.ReLU(),
            nn.Linear(channels // reduction, channels), nn.Sigmoid()
        )
    def forward(self, x):
        w = x.mean(dim=-1)
        return x * self.fc(w).unsqueeze(-1)

class EnhancedCNN1D(nn.Module):
    # REDUCED conv_dim from 256 to 128
    def __init__(self, vocab_size=33, embed_dim=128, conv_dim=128):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.branches = nn.ModuleList()
        for k in [3, 5, 7]:
            self.branches.append(nn.Sequential(
                nn.Conv1d(embed_dim, conv_dim, k, padding=k//2),
                nn.BatchNorm1d(conv_dim), nn.GELU(),
                nn.Conv1d(conv_dim, conv_dim, k, padding=k//2),
                nn.BatchNorm1d(conv_dim), nn.GELU(),
            ))
        self.res_proj = nn.Conv1d(embed_dim, conv_dim, 1)
        self.se = SEBlock(conv_dim * 3)
        self.dropout = nn.Dropout(0.2)
        self.hidden_size = conv_dim * 3 * 2  # Now 768 instead of 1536

    def forward(self, x):
        x = self.embed(x).transpose(1, 2)
        res = self.res_proj(x)
        outs = [branch(x) + res for branch in self.branches]
        combined = torch.cat(outs, dim=1)
        combined = self.se(combined)
        return self.dropout(torch.cat([combined.max(dim=-1).values, combined.mean(dim=-1)], dim=1))

class EnhancedPCP(nn.Module):
    def __init__(self, pcp_dim=30, conv_dim=64, hidden_size=96, num_layers=2):
        super().__init__()
        self.conv_branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(pcp_dim, conv_dim, k, padding=k//2),
                nn.BatchNorm1d(conv_dim), nn.GELU(),
            ) for k in [3, 5, 7]
        ])
        self.conv_merge = nn.Sequential(
            nn.Conv1d(conv_dim * 3, conv_dim * 2, 1),
            nn.BatchNorm1d(conv_dim * 2), nn.GELU(), nn.Dropout1d(0.1)
        )
        self.gru = nn.GRU(conv_dim * 2, hidden_size, num_layers=num_layers,
                          batch_first=True, bidirectional=True,
                          dropout=0.1 if num_layers > 1 else 0)
        self.output_dim = hidden_size * 2  # 192

    def forward(self, pcp_input, attention_mask):
        x = pcp_input.transpose(1, 2)
        x = torch.cat([b(x) for b in self.conv_branches], dim=1)
        x = self.conv_merge(x).transpose(1, 2)
        gru_out, _ = self.gru(x)
        return gru_out

class MultiHeadAttentionPool(nn.Module):
    def __init__(self, dim, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.attn = nn.Sequential(
            nn.Linear(dim, 128), 
            nn.Tanh(), 
            nn.Linear(128, num_heads)
        )
        self._last_weights = None

    def forward(self, x, mask):
        scores = self.attn(x)
        scores = scores.masked_fill(mask.unsqueeze(-1) == 0, -1e4)
        weights = torch.softmax(scores, dim=1)
        self._last_weights = weights
        pooled = (x.unsqueeze(2) * weights.unsqueeze(-1)).sum(dim=1)
        return pooled.view(x.size(0), -1)

    # def orthogonality_loss(self):
    #     if self._last_weights is None:
    #         return 0.0
    #     w = self._last_weights.transpose(1, 2)
    #     gram = torch.bmm(w, w.transpose(1, 2))
    #     eye = torch.eye(self.num_heads, device=gram.device).unsqueeze(0)
    #     return ((gram - eye) ** 2).mean()

    def orthogonality_loss(self):
        """Penalize overlap between head attention distributions."""
        if self._last_weights is None:
            return 0.0
            
        # weights: [B, L, num_heads] -> transpose to [B, num_heads, L]
        w = self._last_weights.transpose(1, 2)  
        
        # Gram matrix of head attention distributions: shape [B, num_heads, num_heads]
        gram = torch.bmm(w, w.transpose(1, 2))  
        
        # Create a boolean mask for the off-diagonal elements (~torch.eye inverts the identity matrix)
        mask = ~torch.eye(self.num_heads, dtype=torch.bool, device=gram.device)
        
        # Penalize only the off-diagonal overlap to encourage heads to focus on different tokens.
        # We want these dot products to be pushed towards 0.
        loss = (gram[:, mask] ** 2).mean()
        
        return loss
    
class GatedFusion(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.gate = nn.Sequential(nn.Linear(input_dim, input_dim), nn.Sigmoid())
    def forward(self, x):
        return x * self.gate(x)

class PeptideNetwork(nn.Module):
    def __init__(self, num_classes=21, mask_token_id=32):
        super().__init__()
        self.mask_token_id = mask_token_id
        self.num_classes = num_classes

        # BOTH encoders are now the 8M parameter t6 model
        self.esm_t6_a = ESM2_Encoder("facebook/esm2_t6_8M_UR50D", trainable=True)        
        self.esm_t6_b = ESM2_Encoder("facebook/esm2_t6_8M_UR50D", trainable=False, unfreeze_last_n=2)                                  
        self.cnn = EnhancedCNN1D()          # Bx768

        # REDUCED cross_dim to 128 to save parameters
        cross_dim = 128
        self.proj_t6_a = nn.Linear(320, cross_dim)
        self.proj_t6_b = nn.Linear(320, cross_dim) # Updated to 320 for t6

        self.cross_t6_a = nn.MultiheadAttention(cross_dim, num_heads=4, batch_first=True, dropout=0.1)
        self.ln_ca_t6_a = nn.LayerNorm(cross_dim)
        
        self.cross_t6_b = nn.MultiheadAttention(cross_dim, num_heads=4, batch_first=True, dropout=0.1)
        self.ln_ca_t6_b = nn.LayerNorm(cross_dim)
        
        self.cross_pcp = nn.MultiheadAttention(cross_dim, num_heads=4, batch_first=True, dropout=0.1)

        self.pool_t6_a = MultiHeadAttentionPool(cross_dim, num_heads=4)
        self.pool_t6_b = MultiHeadAttentionPool(cross_dim, num_heads=4)

        # Bottleneck reduction layer before fusion
        # Concat size: 128*4*3 (pools) + 768 (CNN) = 1536 + 768 = 2304
        concat_size = cross_dim * 4 * 2 + self.cnn.hidden_size
        
        self.dim_reduce = nn.Sequential(
            nn.Linear(concat_size, 512),
            nn.GELU()
        )
        
        # Fusion now operates efficiently on 512 dimensions
        self.fusion = GatedFusion(512)
        self.ln = nn.LayerNorm(512)

        # binary head
        binary_features_dim = 64
        self.binary_features = nn.Sequential(
            nn.Linear(512, binary_features_dim),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        self.binary_classifier = nn.Linear(64, 1) # Final logit

        # Task Query Decoder
        self.task_dim = 128
        self.n_memory_tokens = 16
        self.memory_proj = nn.Sequential(
            nn.Linear(512 + binary_features_dim, self.task_dim * self.n_memory_tokens),
            nn.GELU(),
        )
        self.task_queries = nn.Embedding(num_classes, self.task_dim)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=self.task_dim, nhead=4, dim_feedforward=256,
            batch_first=True, dropout=0.1
        )
        self.task_decoder = nn.TransformerDecoder(decoder_layer, num_layers=2)
        self.task_classifiers = nn.ModuleList([
            nn.Linear(self.task_dim, 1) for _ in range(num_classes)
        ])

    def _mask_tokens(self, input_ids, attention_mask, mask_prob=0.15):
        masked_ids = input_ids.clone()
        prob_matrix = torch.full_like(input_ids, mask_prob, dtype=torch.float)
        prob_matrix[attention_mask == 0] = 0
        prob_matrix[:, 0] = 0
        seq_lens = attention_mask.sum(dim=1)
        for i in range(len(seq_lens)):
            if seq_lens[i] > 1:
                prob_matrix[i, seq_lens[i] - 1] = 0
        mask = torch.bernoulli(prob_matrix).bool()
        masked_ids[mask] = self.mask_token_id
        return masked_ids

    def _extract_features(self, seq_input, seq_mask, pcp_input):
        esm6_a_seq = self.esm_t6_a(seq_input, seq_mask)       
        esm6_b_seq = self.esm_t6_b(seq_input, seq_mask)     
        cnn_feat = self.cnn(seq_input)                          

        t6_a = self.proj_t6_a(esm6_a_seq)       
        t6_b = self.proj_t6_b(esm6_b_seq)    


        kv_pad = seq_mask == 0  

        ca_t6_a, _ = self.cross_t6_a(t6_a, t6_b, t6_b, key_padding_mask=kv_pad)
        ca_t6_a = self.ln_ca_t6_a(t6_a + ca_t6_a)

        ca_t6_b, _ = self.cross_t6_b(t6_b, t6_a, t6_a, key_padding_mask=kv_pad)
        ca_t6_b = self.ln_ca_t6_b(t6_b + ca_t6_b)

        pooled_t6_a = self.pool_t6_a(ca_t6_a, seq_mask)
        pooled_t6_b = self.pool_t6_b(ca_t6_b, seq_mask)

        # Concat -> Reduce -> Fuse
        combined = torch.cat([pooled_t6_a, pooled_t6_b, cnn_feat], dim=1)  
        reduced = self.dim_reduce(combined)
        fusion = self.ln(reduced + self.fusion(reduced))

        binary_features = self.binary_features(fusion)
        
        return binary_features, torch.cat([fusion, binary_features], dim=1)
    
    def _binary_classify(self, binary_features):
        
        binary_logits = self.binary_classifier(binary_features)

        return binary_logits

    def _classify(self, final_fusion):
        B = final_fusion.size(0)
        memory = self.memory_proj(final_fusion).view(B, self.n_memory_tokens, self.task_dim)
        tgt = self.task_queries.weight.unsqueeze(0).expand(B, -1, -1)
        decoded = self.task_decoder(tgt, memory)
        logits = torch.cat([self.task_classifiers[i](decoded[:, i, :])
                           for i in range(self.num_classes)], dim=1)
        return logits

    def forward(self, seq_input, seq_mask, pcp_input, mask_tokens=False):
        if mask_tokens and self.training:
            seq_input = self._mask_tokens(seq_input, seq_mask)
        
        binary_features, combined_features = self._extract_features(seq_input, seq_mask, pcp_input)
        

        return self._binary_classify(binary_features), self._classify(combined_features)


    def ortho_loss(self):
        return (self.pool_t6_a.orthogonality_loss() +
                self.pool_t6_b.orthogonality_loss() 
                ) / 2
    
    def get_features(self, seq_input, seq_mask, pcp_input, mask_tokens=False):
        if mask_tokens and self.training:
            seq_input = self._mask_tokens(seq_input, seq_mask)
        return self._extract_features(seq_input, seq_mask, pcp_input)

    def multi_classify(self, combined):
        return self._classify(combined)
    
    def binary_classify(self, binary_features):
        return self._binary_classify(binary_features)

In [22]:
DEVICE    = 'cuda:1' if torch.cuda.is_available() else 'cpu'

In [23]:
import copy
class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = copy.deepcopy(model)
        self.shadow.eval()
        for p in self.shadow.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        # Update parameters (weights & biases)
        for s_param, m_param in zip(self.shadow.parameters(), model.parameters()):
            s_param.data.mul_(self.decay).add_(m_param.data, alpha=1 - self.decay)
        
        # FIX: Update only floating point buffers (skip int buffers like num_batches_tracked)
        for s_buf, m_buf in zip(self.shadow.buffers(), model.buffers()):
            if s_buf.dtype.is_floating_point:
                s_buf.data.mul_(self.decay).add_(m_buf.data, alpha=1 - self.decay)
            else:
                # For integer buffers, just copy directly (no EMA smoothing needed)
                s_buf.data.copy_(m_buf.data)

    def forward(self, *args, **kwargs):
        return self.shadow(*args, **kwargs)

In [24]:
torch.cuda.empty_cache()

model = PeptideNetwork(num_classes=len(endpoints)-1, mask_token_id=32).to(DEVICE)
ema = EMA(model, decay=0.999)

# --- ASL: Asymmetric Loss for Multi-Label Classification ---
class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip

    def forward(self, logits, targets):
        p = torch.sigmoid(logits)
        pos_part = targets * torch.log(p.clamp(min=1e-8))
        neg_p = (1 - p).clamp(min=1e-8)
        if self.clip > 0:
            neg_p = (neg_p + self.clip).clamp(max=1)
        neg_part = (1 - targets) * torch.log(neg_p)
        pos_weight = (1 - p) ** self.gamma_pos
        neg_weight = p ** self.gamma_neg
        loss = -(pos_weight * pos_part + neg_weight * neg_part)
        return loss.mean()

criterion = AsymmetricLoss(gamma_neg=4, gamma_pos=1, clip=0.05)
binary_criterion = nn.BCEWithLogitsLoss()

# Optimizer — 3 LR groups (Lowered LRs to prevent early overfitting)
esm_t6_ids = set(id(p) for p in model.esm_t6_a.esm_mlm.parameters())
esm_t6_b_ids = set(id(p) for p in model.esm_t6_b.esm_mlm.parameters())

esm_t6_params = [p for p in model.parameters() if id(p) in esm_t6_ids and p.requires_grad]
esm_t6_b_params = [p for p in model.parameters() if id(p) in esm_t6_b_ids and p.requires_grad]
other_params = [p for p in model.parameters() if id(p) not in esm_t6_ids and id(p) not in esm_t6_b_ids and p.requires_grad]

optimizer = optim.AdamW([
    {'params': esm_t6_params, 'lr': 1e-5},     # Reduced from 1e-5
    {'params': esm_t6_b_params, 'lr': 2e-6},   # Reduced from 2e-6
    {'params': other_params, 'lr': 1e-4},      # Reduced from 1e-4
], weight_decay=0.05)                          # Kept the stronger L2 regularization

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total: {total_params/1e6:.1f}M | Trainable: {trainable_params/1e6:.1f}M")
print(f"Loss: ASL(γ-=4, γ+=1, clip=0.05) | v19: Triple CrossAttn + Task Query Decoder")

Total: 18.7M | Trainable: 13.7M
Loss: ASL(γ-=4, γ+=1, clip=0.05) | v19: Triple CrossAttn + Task Query Decoder


In [25]:
NON_FUNC_IDX = endpoint_index['non-functional']
FUNC_INDICES = [i for k, i in endpoint_index.items() if k != 'non-functional']

In [26]:
epochs = 80

In [27]:
# Warmup + Cosine Annealing
steps_per_epoch = len(train_loader)
warmup_epochs = 3
total_steps = epochs * steps_per_epoch

def lr_lambda(step):
    if step < warmup_epochs * steps_per_epoch:
        return step / (warmup_epochs * steps_per_epoch)
    progress = (step - warmup_epochs * steps_per_epoch) / (total_steps - warmup_epochs * steps_per_epoch)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
print(f"Warmup ({warmup_epochs} ep) + CosineAnnealing, total {epochs} epochs, {total_steps} steps")


Warmup (3 ep) + CosineAnnealing, total 80 epochs, 225600 steps


In [28]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score


In [29]:
from sklearn.metrics import matthews_corrcoef, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import json

In [30]:
# --- HELPER FUNCTIONS ---
def calculate_complete_metrics(y_true, y_pred_probs, thresholds, class_names):
    results = {}
    for i in range(y_true.shape[1]):
        y_pred_cls = (y_pred_probs[:, i] >= thresholds[i]).astype(int)
        mcc = matthews_corrcoef(y_true[:, i], y_pred_cls)
        acc = accuracy_score(y_true[:, i], y_pred_cls)
        prec = precision_score(y_true[:, i], y_pred_cls, zero_division=0)
        rec = recall_score(y_true[:, i], y_pred_cls, zero_division=0)
        f1 = f1_score(y_true[:, i], y_pred_cls, zero_division=0)
        try:
            auc_val = roc_auc_score(y_true[:, i], y_pred_probs[:, i])
        except ValueError:
            auc_val = 0.5
        tn, fp, fn, tp = confusion_matrix(y_true[:, i], y_pred_cls, labels=[0, 1]).ravel()
        results[class_names[i]] = {
            "mcc": float(mcc),
            "threshold": float(thresholds[i]),
            "accuracy": float(acc),
            "precision": float(prec),
            "recall": float(rec),
            "f1_score": float(f1),
            "roc_auc": float(auc_val),
            "tp": int(tp),
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
        }
    return results

def find_optimal_thresholds(y_true, y_pred_probs):
    optimal_thresholds = []
    for i in range(y_true.shape[1]):
        best_mcc = -1
        best_threshold = 0.5
        for threshold in np.arange(0.1, 0.95, 0.05):
            y_pred_class = (y_pred_probs[:, i] >= threshold).astype(int)
            mcc = matthews_corrcoef(y_true[:, i], y_pred_class)
            if mcc > best_mcc:
                best_mcc = mcc
                best_threshold = threshold
        optimal_thresholds.append(best_threshold)
    return optimal_thresholds

def calculate_mcc(tp, tn, fp, fn):
    num = (tp * tn) - (fp * fn)
    den_sq = (tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)
    return num / math.sqrt(den_sq) if den_sq > 0 else 0.0

def aggregate_mcc(*metric_groups):
    tp, tn, fp, fn = 0, 0, 0, 0
    for metrics in metric_groups:
        for values in metrics.values():
            tp += values['tp']
            tn += values['tn']
            fp += values['fp']
            fn += values['fn']
    return calculate_mcc(tp, tn, fp, fn)

def feature_mixup(features, labels, alpha=0.4):
    batch_size = features.size(0)
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    index = torch.randperm(batch_size, device=features.device)
    mixed_features = lam * features + (1 - lam) * features[index]
    mixed_labels = lam * labels + (1 - lam) * labels[index]
    return mixed_features, mixed_labels

def rdrop_kl_loss(logits1, logits2):
    kl1 = F.binary_cross_entropy_with_logits(logits1, torch.sigmoid(logits2).detach(), reduction='mean')
    kl2 = F.binary_cross_entropy_with_logits(logits2, torch.sigmoid(logits1).detach(), reduction='mean')
    return (kl1 + kl2) / 2

def enable_dropout(model):
    """
    Specifically tailored for PeptideNetwork to activate all sources of dropout
    during Test-Time Augmentation (TTA), including functional dropouts hidden 
    inside complex PyTorch modules.
    """
    for module in model.modules():
        class_name = module.__class__.__name__
        
        # 1. Standard explicit dropout layers (Dropout, Dropout1d, Dropout2d)
        if class_name.startswith('Dropout'):
            module.train()
            
        # 2. Cross-Attention functional dropout
        elif class_name == 'MultiheadAttention':
            module.train()
            
        # 3. Task Query Decoder functional dropout
        elif class_name in ['TransformerDecoder', 'TransformerDecoderLayer']:
            module.train()
            
        # 4. GRU functional dropout (applied between internal layers)
        elif class_name == 'GRU':
            module.train()


# --- SWA ---
swa_start_epoch = 30
swa_model = None
swa_n = 0

def update_swa(model_state, swa_state, n):
    if swa_state is None:
        return {k: v.clone() for k, v in model_state.items()}, 1
    for k in swa_state:
        swa_state[k] = (swa_state[k] * n + model_state[k]) / (n + 1)
    return swa_state, n + 1

# --- MAIN TRAINING LOOP ---
history_log = []
history_log_path = "best_model.json"
best_model_path = "best_model.pt"
best_val_mcc = -1.0
scaler = torch.amp.GradScaler('cuda')
patience_counter = 0
early_stop_patience = 20
mixup_alpha = 0.4
rdrop_alpha = 1.0
tta_passes = 5

func_names = [index_endpoint[i] for i in FUNC_INDICES]
bin_names = ["NON-FUNCTIONAL"]
bin_thresh = [0.5]
spec_thresh = [0.5] * len(func_names)

print("Training v19: Triple CrossAttn + Task Query Decoder + ASL")
print(f"  ASL(γ-=4, γ+=1, clip=0.05)")
print(f"  R-Drop α={rdrop_alpha} | Mixup α={mixup_alpha} | SWA from ep {swa_start_epoch}")
print(f"  Validation eval: TTA={tta_passes} | Test eval: simple (no TTA)")
print("=" * 70)
for epoch in range(1, epochs + 1):
    model.train()
    total_train_loss = 0.0

    for data in train_loader:
        input_ids = data['input_ids'].to(DEVICE)
        attention_mask = data['attention_mask'].to(DEVICE)
        pcp = data['pcp'].to(DEVICE)
        labels = data['labels'].to(DEVICE)

        optimizer.zero_grad()

        # Binary label: 1 if NON-FUNCTIONAL, 0 if FUNCTIONAL
        target_bin = labels[:, NON_FUNC_IDX].unsqueeze(1).float()
        target_spec = labels[:, FUNC_INDICES]

        with torch.amp.autocast('cuda'):
            # Two augmented passes for R-Drop
            bin_features1, features1 = model.get_features(input_ids, attention_mask, pcp, mask_tokens=True)
            bin_features2, features2 = model.get_features(input_ids, attention_mask, pcp, mask_tokens=True)

            # 1. Clean predictions
            logits1_clean = model.multi_classify(features1)
            logits2_clean = model.multi_classify(features2)

            # 2. Binary classification (Always on the whole batch)
            binary_logits = model.binary_classify(bin_features2)
            binary_loss = binary_criterion(binary_logits, target_bin)
            ortho = model.ortho_loss()

            # --- THE FIX: Boolean Masking ---
            is_functional_mask = (target_bin == 0).view(-1).bool()
            
            if is_functional_mask.sum() > 0:
                # Slice out ONLY the functional peptides
                func_f1 = features1[is_functional_mask]
                func_targets = target_spec[is_functional_mask]
                func_logits1 = logits1_clean[is_functional_mask]
                func_logits2 = logits2_clean[is_functional_mask]

                # R-Drop only on functional
                rdrop_loss = rdrop_kl_loss(func_logits1, func_logits2)
                
                # Mixup only on functional
                mixed_f, mixed_y = feature_mixup(func_f1, func_targets, alpha=mixup_alpha)
                logits_mix = model.multi_classify(mixed_f)
                
                # ASL evaluates ONLY the functional slice
                asl_loss_mix = criterion(logits_mix, mixed_y)
                asl_loss_clean = criterion(func_logits2, func_targets)
                
                spec_loss = (asl_loss_mix + asl_loss_clean) / 2 + (rdrop_alpha * rdrop_loss) + (0.1 * ortho)
                loss = 0.3 * binary_loss + 0.7 * spec_loss
            else:
                # If batch is 100% non-functional
                loss = binary_loss + (0.1 * ortho)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        ema.update(model)

        total_train_loss += loss.item() * input_ids.size(0)

    # --- SWA Update ---
    if epoch >= swa_start_epoch:
        swa_model, swa_n = update_swa(ema.shadow.state_dict(), swa_model, swa_n)

    if swa_model is not None and epoch >= swa_start_epoch:
        eval_model = copy.deepcopy(ema.shadow)
        eval_model.load_state_dict(swa_model)
        eval_model.eval()
    else:
        eval_model = ema.shadow
        eval_model.eval()

    # --- VALIDATION (TTA) ---
    total_val_loss = 0.0
    val_bin_y = []
    val_spec_y = []
    val_tta_bin_preds = [[] for _ in range(tta_passes)]
    val_tta_spec_preds = [[] for _ in range(tta_passes)]

    with torch.no_grad():
        for data in val_loader:
            input_ids = data['input_ids'].to(DEVICE)
            attention_mask = data['attention_mask'].to(DEVICE)
            pcp = data['pcp'].to(DEVICE)
            labels = data['labels'].to(DEVICE)

            target_bin = labels[:, NON_FUNC_IDX].unsqueeze(1).float()
            target_spec = labels[:, FUNC_INDICES]

            with torch.amp.autocast('cuda'):
                pred_bin, pred_spec = eval_model(input_ids, attention_mask, pcp)
                loss_bin = binary_criterion(pred_bin, target_bin)

                is_functional_mask = (target_bin == 0).view(-1).bool()
                if is_functional_mask.sum() > 0:
                    loss_spec = criterion(pred_spec[is_functional_mask], target_spec[is_functional_mask])
                    loss_val = 0.3 * loss_bin + 0.7 * loss_spec
                else:
                    loss_val = loss_bin  # fallback if batch has no functional peptides

            total_val_loss += loss_val.item() * input_ids.size(0)
            
            # Append entire batch for both binary and specific targets
            val_bin_y.append(target_bin.cpu().numpy())
            val_spec_y.append(target_spec.cpu().numpy()) 
            
            val_tta_bin_preds[0].append(torch.sigmoid(pred_bin).detach().cpu().numpy())
            val_tta_spec_preds[0].append(torch.sigmoid(pred_spec).detach().cpu().numpy())

            if tta_passes > 1:
                enable_dropout(eval_model)
                for t in range(1, tta_passes):
                    with torch.amp.autocast('cuda'):
                        pred_bin_t, pred_spec_t = eval_model(input_ids, attention_mask, pcp)
                    val_tta_bin_preds[t].append(torch.sigmoid(pred_bin_t).detach().cpu().numpy())
                    val_tta_spec_preds[t].append(torch.sigmoid(pred_spec_t).detach().cpu().numpy())
                eval_model.eval()

    y_true_bin_val = np.concatenate(val_bin_y)
    y_pred_bin_val = np.mean([np.concatenate(pred_list) for pred_list in val_tta_bin_preds], axis=0)
    
    y_true_spec_val = np.concatenate(val_spec_y)
    raw_pred_spec_val = np.mean([np.concatenate(pred_list) for pred_list in val_tta_spec_preds], axis=0)

    # SOFT GATING: Scale specific predictions by functionality probability
    # y_pred_bin_val is probability of NON-FUNCTIONAL, so 1 - y_pred_bin_val is prob of FUNCTIONAL
    y_pred_spec_val = raw_pred_spec_val * (1 - y_pred_bin_val)

    # Compute optimal thresholds on Validation set
    opt_bin_thresh = find_optimal_thresholds(y_true_bin_val.reshape(-1, 1), y_pred_bin_val.reshape(-1, 1))
    opt_spec_thresh = find_optimal_thresholds(y_true_spec_val, raw_pred_spec_val)


    val_bin_metrics = calculate_complete_metrics(y_true_bin_val, y_pred_bin_val, opt_bin_thresh, bin_names)
    val_spec_metrics = calculate_complete_metrics(y_true_spec_val, y_pred_spec_val, opt_spec_thresh, func_names)
    val_total_mcc = aggregate_mcc(val_bin_metrics, val_spec_metrics)
    
    val_func_mccs = [metrics['mcc'] for metrics in val_spec_metrics.values() if not np.isnan(metrics['mcc'])]
    val_avg_func_mcc = sum(val_func_mccs) / len(val_func_mccs) if val_func_mccs else 0.0


    # --- TEST (TTA) ---
    total_test_loss = 0.0
    test_bin_y = []
    test_spec_y = []
    test_tta_bin_preds = [[] for _ in range(tta_passes)]
    test_tta_spec_preds = [[] for _ in range(tta_passes)]

    with torch.no_grad():
        for data in test_loader:
            input_ids = data['input_ids'].to(DEVICE)
            attention_mask = data['attention_mask'].to(DEVICE)
            pcp = data['pcp'].to(DEVICE)
            labels = data['labels'].to(DEVICE)

            target_bin = labels[:, NON_FUNC_IDX].unsqueeze(1).float()
            target_spec = labels[:, FUNC_INDICES]

            with torch.amp.autocast('cuda'):
                pred_bin, pred_spec = eval_model(input_ids, attention_mask, pcp)
                loss_bin = binary_criterion(pred_bin, target_bin)
                
                is_functional_mask = (target_bin == 0).view(-1).bool()
                if is_functional_mask.sum() > 0:
                    loss_spec = criterion(pred_spec[is_functional_mask], target_spec[is_functional_mask])
                    loss_test = 0.3 * loss_bin + 0.7 * loss_spec
                else:
                    loss_test = loss_bin

            total_test_loss += loss_test.item() * input_ids.size(0)
            
            test_bin_y.append(target_bin.cpu().numpy())
            test_spec_y.append(target_spec.cpu().numpy())
            
            # First pass predictions
            test_tta_bin_preds[0].append(torch.sigmoid(pred_bin).detach().cpu().numpy())
            test_tta_spec_preds[0].append(torch.sigmoid(pred_spec).detach().cpu().numpy())

            # Additional TTA passes
            if tta_passes > 1:
                enable_dropout(eval_model)
                for t in range(1, tta_passes):
                    with torch.amp.autocast('cuda'):
                        pred_bin_t, pred_spec_t = eval_model(input_ids, attention_mask, pcp)
                    test_tta_bin_preds[t].append(torch.sigmoid(pred_bin_t).detach().cpu().numpy())
                    test_tta_spec_preds[t].append(torch.sigmoid(pred_spec_t).detach().cpu().numpy())
                eval_model.eval()

    y_true_bin_test = np.concatenate(test_bin_y)
    y_pred_bin_test = np.mean([np.concatenate(pred_list) for pred_list in test_tta_bin_preds], axis=0)
    
    y_true_spec_test = np.concatenate(test_spec_y)
    raw_pred_spec_test = np.mean([np.concatenate(pred_list) for pred_list in test_tta_spec_preds], axis=0)

    # SOFT GATING for Test
    y_pred_spec_test = raw_pred_spec_test * (1 - y_pred_bin_test)

    test_bin_metrics = calculate_complete_metrics(y_true_bin_test, y_pred_bin_test, opt_bin_thresh, bin_names)
    test_spec_metrics = calculate_complete_metrics(y_true_spec_test, y_pred_spec_test, opt_spec_thresh, func_names)
    test_total_mcc = aggregate_mcc(test_bin_metrics, test_spec_metrics)

    test_func_mccs = [metrics['mcc'] for metrics in test_spec_metrics.values() if not np.isnan(metrics['mcc'])]
    test_avg_func_mcc = sum(test_func_mccs) / len(test_func_mccs) if test_func_mccs else 0.0

    avg_train_loss = total_train_loss / len(train_loader.dataset)
    avg_val_loss = total_val_loss / len(val_loader.dataset)
    avg_test_loss = total_test_loss / len(test_loader.dataset)

    if val_avg_func_mcc > best_val_mcc:
        best_val_mcc = val_avg_func_mcc
        patience_counter = 0
        torch.save(eval_model.state_dict(), best_model_path)
        marker = " >>> BEST <<<"
    else:
        patience_counter += 1
        marker = ""

    lr_now = optimizer.param_groups[0]['lr']
    swa_tag = f" [SWA n={swa_n}]" if epoch >= swa_start_epoch else ""
    print(f"Ep {epoch:02d}/{epochs} | LR: {lr_now:.1e} | Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f} | Test: {avg_test_loss:.4f}{swa_tag}")
    print(f"  MCC -> Func Avg (Val): {val_avg_func_mcc:.4f} | Func Avg (Test): {test_avg_func_mcc:.4f} | Pat: {patience_counter}/{early_stop_patience}{marker}")
    print(f"  Overall Total MCC -> Val: {val_total_mcc:.4f} | Test: {test_total_mcc:.4f}")
    print(f"  NON-FUNCTIONAL MCC -> Val: {val_bin_metrics['NON-FUNCTIONAL']['mcc']:.4f} | Test: {test_bin_metrics['NON-FUNCTIONAL']['mcc']:.4f}")
    
    for name in func_names:
        val_mcc = val_spec_metrics[name]['mcc'] if name in val_spec_metrics else float('nan')
        test_mcc = test_spec_metrics[name]['mcc'] if name in test_spec_metrics else float('nan')
        print(f"  {name:16s}: Val MCC={val_mcc:.4f} | Test MCC={test_mcc:.4f}")
    print("-" * 70)

    epoch_data = {
        "epoch": epoch,
        "train_loss": avg_train_loss,
        "val_loss": avg_val_loss,
        "test_loss": avg_test_loss,
        "val_total_mcc": float(val_total_mcc),
        "test_total_mcc": float(test_total_mcc),
        "thresholds": {
            "binary": opt_bin_thresh,
            "functional": opt_spec_thresh,
        },
        "val_metrics": {**val_bin_metrics, **val_spec_metrics},
        "test_metrics": {**test_bin_metrics, **test_spec_metrics},
    }
    history_log.append(epoch_data)
    with open(history_log_path, "w") as f:
        json.dump(history_log, f, indent=2)

    if patience_counter >= early_stop_patience:
        print(f"\nEarly stopping at epoch {epoch}!")
        break

    if swa_model is not None and epoch >= swa_start_epoch:
        del eval_model

print(f"\nTraining Complete! Best Val MCC: {best_val_mcc:.4f}")

Training v19: Triple CrossAttn + Task Query Decoder + ASL
  ASL(γ-=4, γ+=1, clip=0.05)
  R-Drop α=1.0 | Mixup α=0.4 | SWA from ep 30
  Validation eval: TTA=5 | Test eval: simple (no TTA)
Ep 02/80 | LR: 6.7e-06 | Train: 0.5214 | Val: 0.0591 | Test: 0.0580
  MCC -> Func Avg (Val): 0.3834 | Func Avg (Test): 0.3837 | Pat: 0/20 >>> BEST <<<
  Overall Total MCC -> Val: 0.7431 | Test: 0.7347
  NON-FUNCTIONAL MCC -> Val: 0.9311 | Test: 0.9286
  anti-bacterial  : Val MCC=0.6962 | Test MCC=0.7168
  anti-cancer     : Val MCC=0.3530 | Test MCC=0.3625
  anti-fungal     : Val MCC=0.3779 | Test MCC=0.4234
  anti-parasitic  : Val MCC=0.3186 | Test MCC=0.3637
  anti-viral      : Val MCC=0.3149 | Test MCC=0.3514
  cell-cell-communication: Val MCC=0.1368 | Test MCC=0.2206
  drug-delivery   : Val MCC=0.2416 | Test MCC=0.1457
  immunological   : Val MCC=0.3844 | Test MCC=0.3620
  inhibitor       : Val MCC=0.2842 | Test MCC=0.2807
  metabolic       : Val MCC=0.3424 | Test MCC=0.2738
  other-functional: Val 